            # Round 5: The Final Stretch

            This notebook documents the final `trader.py` submission for the 50
            new Round 5 products plus the Ignith manual Ashflow Alpha allocation.

            ## Final replay summary

            - Combined deterministic replay: **888,469.0 XIRECS**
            - Fill model: book-crossing plus public-repo-style passive fills from
              bot trades that would have interacted with our improved quote.
            - Fill count: 5,346 crossing fills, 13,215 passive fills, 55,061 total filled units
            - Active algorithmic file: `trader.py`
            - Diagnostics script: `scripts/round5_diagnostics.py`

            | Day | Replay PnL |
|---:|---:|
| 2 | 170,990.5 |
| 3 | 114,158.0 |
| 4 | 603,320.5 |

            ## Top replay contributors

            | Product | Replay PnL |
|---|---:|
| `ROBOT_DISHES` | 438,118.0 |
| `OXYGEN_SHAKE_EVENING_BREATH` | 59,868.0 |
| `OXYGEN_SHAKE_CHOCOLATE` | 53,964.0 |
| `PEBBLES_S` | 43,858.0 |
| `PEBBLES_XL` | 31,424.0 |
| `PANEL_1X4` | 27,843.0 |
| `SNACKPACK_STRAWBERRY` | 20,858.0 |
| `PANEL_2X4` | 19,900.0 |
| `OXYGEN_SHAKE_MORNING_BREATH` | 18,762.0 |
| `SLEEP_POD_COTTON` | 18,361.0 |
| `UV_VISOR_AMBER` | 17,874.5 |
| `UV_VISOR_YELLOW` | 16,897.0 |
| `TRANSLATOR_VOID_BLUE` | 16,212.0 |
| `SNACKPACK_PISTACHIO` | 15,155.0 |
| `MICROCHIP_SQUARE` | 15,051.5 |
| `UV_VISOR_ORANGE` | 15,035.0 |

## Public IMC repository lessons used

I reviewed public Prosperity writeups and code repositories before
finalizing the Round 5 shape:

| Source | Applicable lesson |
|---|---|
| [Frankfurt Hedgehogs, Prosperity 3, 2nd globally](https://github.com/TimoDiehm/imc-prosperity-3) | Treat Prosperity as a microstructure game first: identify the fair price, improve inside the spread, and use dashboards/backtests to inspect fills. Their writeup emphasizes WallMid/true-price reasoning, inventory clearing, and bot behavior. |
| [Linear Utility, Prosperity 2, 2nd place](https://github.com/ericcccsliu/imc-prosperity-2) | Build a replay harness, grid search simple parameters, and prefer structural edges over decorative correlations. Their most durable edges came from true-price market making, conversion arbitrage, and spread trades. |
| [jmerle, Prosperity 2, 9th overall](https://github.com/jmerle/imc-prosperity-2) | In Round 5, de-anonymized flow can dominate. Their writeup found named-trader directional signals and also warns that overfit directional products can lose badly. |
| [AlphaBaguette, Prosperity 3](https://github.com/Sylvain-Topeza/imc-prosperity-3) | Combine complementary small edges: adaptive market making, informed flow when available, index/spread logic, and strict position limits. |
| [Prosperity preparation discussion](https://www.reddit.com/r/learnquant/comments/1rvf93p/how_to_actually_compete_and_maybe_win_in_imc/) | The common archetypes are fixed-fair market making, basket/stat-arb spreads, options, location arbitrage, and Round 5 trader-ID flow. |

The Round 5 trade files in this dataset do **not** reveal buyer/seller
IDs; every buyer and seller field is blank. Therefore the prior
"copy the informed trader" trick is not directly available here. The
final algorithm instead applies the reusable parts of those writeups:
fair-price market making, fill-aware parameter selection, and only a
few high-confidence directional overlays.

## Local data findings

The zip contains three historical days: days 2, 3, and 4. Each day has
10,000 timestamps for all 50 products.

Main discoveries:

- The strongest repeatable edge is passive inside-spread market making
  on selected products. The replay only enables a product when it is
  profitable on all three provided days under a conservative bot-trade
  fill model.
- `ROBOT_DISHES`, `OXYGEN_SHAKE_CHOCOLATE`, and
  `OXYGEN_SHAKE_EVENING_BREATH` have jump-reversion events large
  enough to justify crossing the spread. These are deliberately gated
  by a 30-XIREC one-tick move threshold.
- Snack packs have a useful group-level sum reversion. The algorithm
  trades an 8-lot group target only when the five-product sum is more
  than 150 XIRECS away from its historical anchor.
- Pebbles have a near-exact five-product sum around 50,000, but the
  multi-leg edge is too thin after spread cost. The final bot keeps the
  strong individual `PEBBLES_S` and `PEBBLES_XL` makers and skips the
  group overlay.
- Products without robust replay contribution are left idle. Unused
  symbols are better than forced variance.

In [ ]:
import json
from pathlib import Path

diagnostics = json.loads(Path("logs/round5_diagnostics.json").read_text())
diagnostics["combined_pnl"], diagnostics["fills"]

In [ ]:
import pandas as pd

totals = diagnostics["per_product_totals"]
pd.Series(totals).sort_values(ascending=False).head(20).to_frame("replay_pnl")

## Trader implementation

The final `trader.py` is Round 5 only. It does not trade any products
from previous rounds.

Strategy layers:

- **Passive selected makers:** quote one tick better than the best
  displayed bid/ask only when the quote still has positive edge to the
  current book mid after inventory skew.
- **Jump-reversion takers:** cross only after very large one-tick
  moves in the three products where this paid across replay.
- **Snackpack sum overlay:** hold a small common target across the five
  snack packs when their aggregate level is materially too high or too
  low.
- **Risk controls:** all logic respects the hard 10-unit position
  limit per product, and products without robust evidence are idle.

## Ignith manual strategy

Submit the following Ashflow Alpha manual orders:

| Good | Side | % |
|---|---:|---:|
| Sulfur reactor | Buy | 16% |
| Thermalite core | Buy | 14% |
| Lava cake | Sell | 13% |
| Pyroflex cells | Sell | 11% |
| Magma ink | Buy | 8% |
| Ashes of the Phoenix | Sell | 6% |
| Volcanic incense | Buy | 5% |
| Scoria paste | Buy | 4% |
| Obsidian cutlery | Buy | 3% |

This uses 80% of the manual budget and pays 89,200 XIRECS in fees.
The fee rule makes each product's break-even move equal to its
allocation percentage, so the unused 20% is intentional. It avoids
forcing capital into weaker headlines where the quadratic fee can
overwhelm the news edge.